In [1]:
# CELL 1: Imports, options d'affichage et fonctions des graphiques

import pandas as pd
import numpy as np
import joblib
import shap
from xgboost import XGBClassifier
from tensorflow.keras.models import load_model
from sklearn.metrics import f1_score

# Affichage des tableaux : toutes les colonnes, sur une largeur de 200 caractères
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Fonctions des graphiques Plotly et de préparation des données brutes (graphiques.py, à la racine du projet,
# partagé avec l'application Streamlit)
%run ../graphiques.py

# Dossiers : données préparées par 02, exports pour l'application (créé avant ce notebook), figures statiques
dossier = '../data/output/'
dossier_app = '../data/output/app/'
dossier_figures = '../figures/app/'

In [2]:
# CELL 2: Chargement des données brutes, des artefacts du pré-traitement et des paramètres des modèles

brut = pd.read_csv('../data/raw/weatherAUS.csv')
artefacts = joblib.load('../models/preprocessing_artifacts.joblib')
# Paramètres, options de déséquilibre et seuils retenus en 03 (CELL 61 pour J+1, CELL 121 pour J+2)
parametres_J1 = joblib.load('../models/modeles_parametres.joblib')
parametres_J2 = joblib.load('../models/modeles_parametres_J2.joblib')

print(f"Données brutes: {brut.shape[0]} lignes × {brut.shape[1]} colonnes")
print(f"Date de coupure entraînement / test: {str(artefacts['date_coupure'])[:10]}")
print(f"Seuils retenus J+1: {parametres_J1['seuil_retenu']}")
print(f"Seuils retenus J+2: {parametres_J2['seuil_retenu']}")

Données brutes: 145460 lignes × 23 colonnes
Date de coupure entraînement / test: 2015-11-10
Seuils retenus J+1: {'Régression logistique': np.float64(0.594), 'Random Forest': np.float64(0.438), 'XGBoost': np.float64(0.302), 'KNN': np.float64(0.401), 'Réseau de neurones': np.float64(0.306)}
Seuils retenus J+2: {'Régression logistique': np.float64(0.468), 'Random Forest': np.float64(0.461), 'XGBoost': np.float64(0.373), 'KNN': np.float64(0.201), 'Réseau de neurones': np.float64(0.459)}


In [3]:
# CELL 3: J+1 : F1 de chaque modèle sur chaque pli, paramètres par défaut (1.a)

# Sorties de 03_modelisation : CELL 10 à CELL 14 (F1 affichés à 4 décimales) ;
# modèle naïf (toujours « pas de pluie ») : CELL 7 (F1 = 0 par définition)
scores_plis_J1 = pd.DataFrame({
    'Pli': [1, 2, 3, 4, 5],
    'Régression logistique': [0.5689, 0.6071, 0.5629, 0.6011, 0.5437],
    'Random Forest': [0.5592, 0.609, 0.5836, 0.6192, 0.5614],
    'XGBoost': [0.5567, 0.6395, 0.6144, 0.6453, 0.5924],
    'KNN': [0.4839, 0.568, 0.5422, 0.5746, 0.5312],
    'Réseau de neurones': [0.5876, 0.6427, 0.5988, 0.6305, 0.5865],
    'Modèle naïf': [0.0, 0.0, 0.0, 0.0, 0.0]
})
scores_plis_J1.to_csv(dossier_app + 'scores_plis_J1.csv', index=False)
print(scores_plis_J1)

   Pli  Régression logistique  Random Forest  XGBoost     KNN  Réseau de neurones  Modèle naïf
0    1                 0.5689         0.5592   0.5567  0.4839              0.5876          0.0
1    2                 0.6071         0.6090   0.6395  0.5680              0.6427          0.0
2    3                 0.5629         0.5836   0.6144  0.5422              0.5988          0.0
3    4                 0.6011         0.6192   0.6453  0.5746              0.6305          0.0
4    5                 0.5437         0.5614   0.5924  0.5312              0.5865          0.0


In [4]:
# CELL 4: J+2 : F1 de chaque modèle sur chaque pli, paramètres par défaut (1.a)

# Sorties de 03_modelisation : CELL 80 à CELL 84 (F1 affichés à 4 décimales) ;
# modèle naïf (toujours « pas de pluie ») : CELL 79 (F1 = 0 par définition)
scores_plis_J2 = pd.DataFrame({
    'Pli': [1, 2, 3, 4, 5],
    'Régression logistique': [0.1377, 0.2231, 0.177, 0.2204, 0.1568],
    'Random Forest': [0.1282, 0.2293, 0.241, 0.2527, 0.1891],
    'XGBoost': [0.1946, 0.3178, 0.315, 0.3412, 0.2562],
    'KNN': [0.2381, 0.3108, 0.3215, 0.3212, 0.2939],
    'Réseau de neurones': [0.1662, 0.194, 0.2418, 0.3181, 0.2461],
    'Modèle naïf': [0.0, 0.0, 0.0, 0.0, 0.0]
})
scores_plis_J2.to_csv(dossier_app + 'scores_plis_J2.csv', index=False)
print(scores_plis_J2)

   Pli  Régression logistique  Random Forest  XGBoost     KNN  Réseau de neurones  Modèle naïf
0    1                 0.1377         0.1282   0.1946  0.2381              0.1662          0.0
1    2                 0.2231         0.2293   0.3178  0.3108              0.1940          0.0
2    3                 0.1770         0.2410   0.3150  0.3215              0.2418          0.0
3    4                 0.2204         0.2527   0.3412  0.3212              0.3181          0.0
4    5                 0.1568         0.1891   0.2562  0.2939              0.2461          0.0


In [5]:
# CELL 5: J+1 : F1 moyen de chaque modèle à chaque étape (1.b)

# Sorties de 03_modelisation (pleine précision) :
# - paramètres par défaut : CELL 16 (moyenne sur les 5 plis)
# - option de déséquilibre retenue : CELL 25 (option retenue par modèle en CELL 26)
# - hyperparamètres : CELL 37 (meilleure combinaison retenue pour Random Forest, XGBoost et le réseau) ;
#   régression logistique et KNN ne sont pas optimisés : valeur de l'étape précédente (CELL 35)
# - seuil de décision : CELL 48 (F1 moyen au seuil retenu sur les plis de validation)
# - test : CELL 58 (F1 sur l'ensemble de test, seuil retenu)
progression_J1 = pd.DataFrame({
    'Modèle': ['Régression logistique', 'Random Forest', 'XGBoost', 'KNN', 'Réseau de neurones'],
    'Paramètres par défaut': [0.5767435236542883, 0.5864700117470617, 0.6096575149112418, 0.5399913327251544, 0.6092436988237679],
    'Option de déséquilibre': [0.6088145247352227, 0.6294550410705986, 0.6108966339341804, 0.5624979708679176, 0.6092436988237679],
    'Hyperparamètres': [0.6088145247352227, 0.6331325881138113, 0.6202130271607876, 0.5624979708679176, 0.61076258049905],
    'Seuil de décision': [0.6216882458298927, 0.6380597375409707, 0.6450024484049643, 0.5624979708679176, 0.6441794780414264],
    'Test': [0.6154065620542083, 0.6418578225978269, 0.6595914807007341, 0.5687696398356297, 0.6503307276007216]
})
progression_J1.to_csv(dossier_app + 'progression_J1.csv', index=False)
print(progression_J1)

                  Modèle  Paramètres par défaut  Option de déséquilibre  Hyperparamètres  Seuil de décision      Test
0  Régression logistique               0.576744                0.608815         0.608815           0.621688  0.615407
1          Random Forest               0.586470                0.629455         0.633133           0.638060  0.641858
2                XGBoost               0.609658                0.610897         0.620213           0.645002  0.659591
3                    KNN               0.539991                0.562498         0.562498           0.562498  0.568770
4     Réseau de neurones               0.609244                0.609244         0.610763           0.644179  0.650331


In [6]:
# CELL 6: J+2 : F1 moyen de chaque modèle à chaque étape (1.b)

# Sorties de 03_modelisation (pleine précision) :
# - paramètres par défaut : CELL 85 (moyenne sur les 5 plis)
# - option de déséquilibre retenue : CELL 93 (option retenue par modèle en CELL 94)
# - hyperparamètres : CELL 100 (meilleure combinaison retenue pour Random Forest, XGBoost et le réseau) ;
#   régression logistique et KNN ne sont pas optimisés : valeur de l'étape précédente (CELL 35 et CELL 100)
# - seuil de décision : CELL 108 (F1 moyen au seuil retenu sur les plis de validation)
# - test : CELL 118 (F1 sur l'ensemble de test, seuil retenu)
progression_J2 = pd.DataFrame({
    'Modèle': ['Régression logistique', 'Random Forest', 'XGBoost', 'KNN', 'Réseau de neurones'],
    'Paramètres par défaut': [0.18299921406398606, 0.20807013101881583, 0.28495645306486844, 0.29706486590131415, 0.23322786353485844],
    'Option de déséquilibre': [0.44516053869625394, 0.36682338162569633, 0.29744848484754505, 0.4133460403456325, 0.42854094231642115],
    'Hyperparamètres': [0.44516053869625394, 0.45228584888412754, 0.3516054840292001, 0.4133460403456325, 0.45861791693366083],
    'Seuil de décision': [0.44775900145331116, 0.45814904816032553, 0.43821896333102195, 0.4196914164353469, 0.4621308704429318],
    'Test': [0.4511564051114162, 0.4627670396744659, 0.43547348078161907, 0.4208604257593302, 0.4709464341005264]
})
progression_J2.to_csv(dossier_app + 'progression_J2.csv', index=False)
print(progression_J2)

                  Modèle  Paramètres par défaut  Option de déséquilibre  Hyperparamètres  Seuil de décision      Test
0  Régression logistique               0.182999                0.445161         0.445161           0.447759  0.451156
1          Random Forest               0.208070                0.366823         0.452286           0.458149  0.462767
2                XGBoost               0.284956                0.297448         0.351605           0.438219  0.435473
3                    KNN               0.297065                0.413346         0.413346           0.419691  0.420860
4     Réseau de neurones               0.233228                0.428541         0.458618           0.462131  0.470946


In [7]:
# CELL 7: J+1 : résultats de chaque modèle sur l'ensemble de test (1.c)

# Sorties de 03_modelisation : CELL 58 (F1 en pleine précision, autres valeurs à 4 décimales) et CELL 61
# (taille des fichiers en Mo, 2 décimales) ; modèle naïf : pas de seuil, d'AUC, de durée ni de fichier
lignes = [
    ['XGBoost', 0.302, 0.6595914807007341, 0.6124, 0.7146, 0.8354, 0.8823, 7.9468, 0.049, 6.98],
    ['Réseau de neurones', 0.306, 0.6503307276007216, 0.622, 0.6814, 0.8365, 0.8777, 68.2, 1.5583, 0.16],
    ['Random Forest', 0.438, 0.6418578225978269, 0.5845, 0.7118, 0.8227, 0.8745, 8.7376, 0.319, 353.21],
    ['Régression logistique', 0.594, 0.6154065620542083, 0.5624, 0.6795, 0.8104, 0.8504, 0.7326, 0.0135, 0.0],
    ['KNN', 0.401, 0.5687696398356297, 0.4614, 0.7412, 0.7491, 0.8089, 0.0716, 5.8678, 33.65],
    ['Naïf (toujours pas de pluie)', np.nan, 0.0, 0.0, 0.0, 0.7768, np.nan, np.nan, np.nan, np.nan]
]
resultats_test_J1 = pd.DataFrame(lignes, columns=['Modèle', 'Seuil', 'F1', 'Précision', 'Rappel', 'Exactitude', 'AUC',
                                               'Temps entraînement (s)', 'Temps prédiction (s)',
                                               'Taille du fichier (Mo)'])
resultats_test_J1.to_csv(dossier_app + 'resultats_test_J1.csv', index=False)
print(resultats_test_J1)

                         Modèle  Seuil        F1  Précision  Rappel  Exactitude     AUC  Temps entraînement (s)  Temps prédiction (s)  Taille du fichier (Mo)
0                       XGBoost  0.302  0.659591     0.6124  0.7146      0.8354  0.8823                  7.9468                0.0490                    6.98
1            Réseau de neurones  0.306  0.650331     0.6220  0.6814      0.8365  0.8777                 68.2000                1.5583                    0.16
2                 Random Forest  0.438  0.641858     0.5845  0.7118      0.8227  0.8745                  8.7376                0.3190                  353.21
3         Régression logistique  0.594  0.615407     0.5624  0.6795      0.8104  0.8504                  0.7326                0.0135                    0.00
4                           KNN  0.401  0.568770     0.4614  0.7412      0.7491  0.8089                  0.0716                5.8678                   33.65
5  Naïf (toujours pas de pluie)    NaN  0.000000    

In [8]:
# CELL 8: J+2 : résultats de chaque modèle sur l'ensemble de test (1.c)

# Sorties de 03_modelisation : CELL 118 (F1 en pleine précision, autres valeurs à 4 décimales) et CELL 121
# (taille des fichiers en Mo, 2 décimales) ; modèle naïf : pas de seuil, d'AUC, de durée ni de fichier
lignes = [
    ['Réseau de neurones', 0.459, 0.4709464341005264, 0.3464, 0.7354, 0.6349, 0.734, 137.832, 1.3303, 0.07],
    ['Random Forest', 0.461, 0.4627670396744659, 0.3381, 0.7331, 0.6238, 0.7251, 3.9191, 0.1595, 37.12],
    ['Régression logistique', 0.468, 0.4511564051114162, 0.3355, 0.6885, 0.6298, 0.7063, 0.4381, 0.0079, 0.0],
    ['XGBoost', 0.373, 0.43547348078161907, 0.3265, 0.6537, 0.6255, 0.6875, 0.6072, 0.0098, 0.12],
    ['KNN', 0.201, 0.4208604257593302, 0.2908, 0.7615, 0.5369, 0.6591, 0.0554, 4.1026, 33.35],
    ['Naïf (toujours pas de pluie)', np.nan, 0.0, 0.0, 0.0, 0.779, np.nan, np.nan, np.nan, np.nan]
]
resultats_test_J2 = pd.DataFrame(lignes, columns=['Modèle', 'Seuil', 'F1', 'Précision', 'Rappel', 'Exactitude', 'AUC',
                                               'Temps entraînement (s)', 'Temps prédiction (s)',
                                               'Taille du fichier (Mo)'])
resultats_test_J2.to_csv(dossier_app + 'resultats_test_J2.csv', index=False)
print(resultats_test_J2)

                         Modèle  Seuil        F1  Précision  Rappel  Exactitude     AUC  Temps entraînement (s)  Temps prédiction (s)  Taille du fichier (Mo)
0            Réseau de neurones  0.459  0.470946     0.3464  0.7354      0.6349  0.7340                137.8320                1.3303                    0.07
1                 Random Forest  0.461  0.462767     0.3381  0.7331      0.6238  0.7251                  3.9191                0.1595                   37.12
2         Régression logistique  0.468  0.451156     0.3355  0.6885      0.6298  0.7063                  0.4381                0.0079                    0.00
3                       XGBoost  0.373  0.435473     0.3265  0.6537      0.6255  0.6875                  0.6072                0.0098                    0.12
4                           KNN  0.201  0.420860     0.2908  0.7615      0.5369  0.6591                  0.0554                4.1026                   33.35
5  Naïf (toujours pas de pluie)    NaN  0.000000    

In [9]:
# CELL 9: Fonctions de chargement des modèles finaux et de calcul des probabilités de test (1.d)

def charger_modeles(suffixe):
    # Les 5 modèles finaux sauvegardés par 03_modelisation (suffixe '' pour J+1, '_J2' pour J+2)
    modeles_finaux = {}
    modeles_finaux['Régression logistique'] = joblib.load('../models/modele_regression_logistique' + suffixe + '.joblib')
    modeles_finaux['Random Forest'] = joblib.load('../models/modele_random_forest' + suffixe + '.joblib')
    modeles_finaux['XGBoost'] = XGBClassifier()
    modeles_finaux['XGBoost'].load_model('../models/modele_xgboost' + suffixe + '.json')
    modeles_finaux['KNN'] = joblib.load('../models/modele_knn' + suffixe + '.joblib')
    modeles_finaux['Réseau de neurones'] = load_model('../models/modele_reseau_neurones' + suffixe + '.keras')
    return modeles_finaux


def probabilites_test(modeles_finaux, X_scaled, X_arbres):
    # Probabilité de pluie : données mises à l'échelle pour la régression logistique, KNN et le réseau de neurones
    # (2e colonne de predict pour le réseau), données non mises à l'échelle pour Random Forest et XGBoost
    probas = {}
    probas['Régression logistique'] = modeles_finaux['Régression logistique'].predict_proba(X_scaled)[:, 1]
    probas['Random Forest'] = modeles_finaux['Random Forest'].predict_proba(X_arbres)[:, 1]
    probas['XGBoost'] = modeles_finaux['XGBoost'].predict_proba(X_arbres)[:, 1]
    probas['KNN'] = modeles_finaux['KNN'].predict_proba(X_scaled)[:, 1]
    probas['Réseau de neurones'] = modeles_finaux['Réseau de neurones'].predict(X_scaled.values, verbose=0)[:, 1]
    return probas

In [10]:
# CELL 10: J+1 : probabilité de pluie de chaque modèle pour chaque jour du test (1.d)

X_test_scaled_J1 = pd.read_csv(dossier + 'X_test_J1_scaled.csv')
X_test_arbres_J1 = pd.read_csv(dossier + 'X_test_J1_arbres.csv')
meta_test_J1 = pd.read_csv(dossier + 'meta_test_J1.csv')
y_test_J1 = pd.read_csv(dossier + 'y_test_J1.csv')['RainTomorrow']

modeles_J1 = charger_modeles('')
probas = probabilites_test(modeles_J1, X_test_scaled_J1, X_test_arbres_J1)

# Une ligne par jour du test : cible réelle, station, date, puis la probabilité de pluie de chaque modèle
probas_J1 = pd.DataFrame({'Cible': y_test_J1, 'Location': meta_test_J1['Location'], 'Date': meta_test_J1['Date']})
for nom in MODELES:
    probas_J1[nom] = probas[nom]
probas_J1.to_csv(dossier_app + 'probas_test_J1.csv', index=False)
print(f"probas_test_J1.csv: {probas_J1.shape[0]} lignes × {probas_J1.shape[1]} colonnes")
print(probas_J1.head())

probas_test_J1.csv: 28445 lignes × 8 colonnes
   Cible    Location        Date  Régression logistique  Random Forest   XGBoost  KNN  Réseau de neurones
0      0       Cobar  2015-11-10               0.049361       0.022965  0.003234  0.0            0.002619
1      0    Adelaide  2015-11-10               0.311671       0.244911  0.101016  0.0            0.154665
2      0  Wollongong  2015-11-10               0.841627       0.481225  0.241028  0.6            0.354706
3      1  Launceston  2015-11-10               0.314513       0.160029  0.232890  0.2            0.188322
4      0   GoldCoast  2015-11-10               0.410493       0.201437  0.178825  0.8            0.217879


In [11]:
# CELL 11: J+2 : probabilité de pluie de chaque modèle pour chaque jour du test (1.d)

X_test_scaled_J2 = pd.read_csv(dossier + 'X_test_J2_scaled.csv')
X_test_arbres_J2 = pd.read_csv(dossier + 'X_test_J2_arbres.csv')
meta_test_J2 = pd.read_csv(dossier + 'meta_test_J2.csv')
y_test_J2 = pd.read_csv(dossier + 'y_test_J2.csv')['RainInTwoDays']

modeles_J2 = charger_modeles('_J2')
probas = probabilites_test(modeles_J2, X_test_scaled_J2, X_test_arbres_J2)

# Une ligne par jour du test : cible réelle, station, date, puis la probabilité de pluie de chaque modèle
probas_J2 = pd.DataFrame({'Cible': y_test_J2, 'Location': meta_test_J2['Location'], 'Date': meta_test_J2['Date']})
for nom in MODELES:
    probas_J2[nom] = probas[nom]
probas_J2.to_csv(dossier_app + 'probas_test_J2.csv', index=False)
print(f"probas_test_J2.csv: {probas_J2.shape[0]} lignes × {probas_J2.shape[1]} colonnes")
print(probas_J2.head())

probas_test_J2.csv: 28078 lignes × 8 colonnes
   Cible    Location        Date  Régression logistique  Random Forest   XGBoost  KNN  Réseau de neurones
0      0       Cobar  2015-11-10               0.229728       0.267561  0.212008  1.0            0.394378
1      0    Adelaide  2015-11-10               0.305046       0.337176  0.194562  0.6            0.457200
2      1  Wollongong  2015-11-10               0.615787       0.589750  0.456494  0.4            0.602978
3      1  Launceston  2015-11-10               0.391487       0.342743  0.282020  1.0            0.387997
4      0   GoldCoast  2015-11-10               0.454912       0.465863  0.314565  1.0            0.551576


In [12]:
# CELL 12: Vérification : F1 recalculé à partir des fichiers de probabilités et des seuils retenus (1.d)

def verifier_f1(probas, resultats, seuils, horizon):
    # F1 de chaque modèle au seuil retenu, comparé au F1 de test de 03 (tableau de CELL 7 ou CELL 8)
    for nom in MODELES:
        f1_fichier = f1_score(probas['Cible'], (probas[nom] >= seuils[nom]).astype(int))
        f1_03 = resultats.loc[resultats['Modèle'] == nom, 'F1'].values[0]
        print(f"{horizon} | {nom} | seuil = {seuils[nom]:.3f} | F1 recalculé = {f1_fichier} | F1 de 03 = {f1_03} | "
              f"identiques: {f1_fichier == f1_03}")


verifier_f1(probas_J1, resultats_test_J1, parametres_J1['seuil_retenu'], 'J+1')
print()
verifier_f1(probas_J2, resultats_test_J2, parametres_J2['seuil_retenu'], 'J+2')

J+1 | Régression logistique | seuil = 0.594 | F1 recalculé = 0.6154065620542083 | F1 de 03 = 0.6154065620542083 | identiques: True
J+1 | Random Forest | seuil = 0.438 | F1 recalculé = 0.6418578225978269 | F1 de 03 = 0.6418578225978269 | identiques: True
J+1 | XGBoost | seuil = 0.302 | F1 recalculé = 0.6595914807007341 | F1 de 03 = 0.6595914807007341 | identiques: True
J+1 | KNN | seuil = 0.401 | F1 recalculé = 0.5687696398356297 | F1 de 03 = 0.5687696398356297 | identiques: True
J+1 | Réseau de neurones | seuil = 0.306 | F1 recalculé = 0.6503307276007216 | F1 de 03 = 0.6503307276007216 | identiques: True

J+2 | Régression logistique | seuil = 0.468 | F1 recalculé = 0.4511564051114162 | F1 de 03 = 0.4511564051114162 | identiques: True
J+2 | Random Forest | seuil = 0.461 | F1 recalculé = 0.4627670396744659 | F1 de 03 = 0.4627670396744659 | identiques: True
J+2 | XGBoost | seuil = 0.373 | F1 recalculé = 0.43547348078161907 | F1 de 03 = 0.43547348078161907 | identiques: True


J+2 | KNN | seuil = 0.201 | F1 recalculé = 0.4208604257593302 | F1 de 03 = 0.4208604257593302 | identiques: True
J+2 | Réseau de neurones | seuil = 0.459 | F1 recalculé = 0.4709464341005264 | F1 de 03 = 0.4709464341005264 | identiques: True


In [13]:
# CELL 13: Importance des variables des modèles XGBoost finaux (1.e)

# feature_importances_ des modèles sauvegardés, triées par ordre décroissant
importances_J1 = pd.DataFrame({'Variable': X_test_arbres_J1.columns,
                               'Importance': modeles_J1['XGBoost'].feature_importances_})
importances_J1 = importances_J1.sort_values('Importance', ascending=False)
importances_J2 = pd.DataFrame({'Variable': X_test_arbres_J2.columns,
                               'Importance': modeles_J2['XGBoost'].feature_importances_})
importances_J2 = importances_J2.sort_values('Importance', ascending=False)

importances_J1.to_csv(dossier_app + 'importances_xgboost_J1.csv', index=False)
importances_J2.to_csv(dossier_app + 'importances_xgboost_J2.csv', index=False)
print("J+1, 5 premières variables:")
print(importances_J1.head())
print("\nJ+2, 5 premières variables:")
print(importances_J2.head())

J+1, 5 premières variables:
          Variable  Importance
6      Humidity3pm    0.164236
1         Rainfall    0.093503
15      Season_sin    0.083263
22  WindDir3pm_cos    0.081545
20  WindDir9am_cos    0.081267

J+2, 5 premières variables:
           Variable  Importance
15       Season_sin    0.153509
18  WindGustDir_cos    0.097770
22   WindDir3pm_cos    0.090034
16       Season_cos    0.081663
21   WindDir3pm_sin    0.076942


In [14]:
# CELL 14: Odds ratios des régressions logistiques finales (1.e)

# Odds ratio = exp(coefficient) ; variables mises à l'échelle (mêmes colonnes que les fichiers mis à l'échelle)
coefficients_J1 = modeles_J1['Régression logistique'].coef_[0]
odds_J1 = pd.DataFrame({'Variable': X_test_scaled_J1.columns, 'Coefficient': coefficients_J1,
                        'Odds ratio': np.exp(coefficients_J1)}).sort_values('Odds ratio', ascending=False)
coefficients_J2 = modeles_J2['Régression logistique'].coef_[0]
odds_J2 = pd.DataFrame({'Variable': X_test_scaled_J2.columns, 'Coefficient': coefficients_J2,
                        'Odds ratio': np.exp(coefficients_J2)}).sort_values('Odds ratio', ascending=False)

odds_J1.to_csv(dossier_app + 'odds_ratios_J1.csv', index=False)
odds_J2.to_csv(dossier_app + 'odds_ratios_J2.csv', index=False)
print("J+1, 3 plus grands odds ratios:")
print(odds_J1.head(3))
print("\nJ+2, 3 plus grands odds ratios:")
print(odds_J2.head(3))

J+1, 3 plus grands odds ratios:
        Variable  Coefficient  Odds ratio
6    Humidity3pm     0.956480    2.602520
2  WindGustSpeed     0.813155    2.255011
5    Humidity9am     0.521890    1.685210

J+2, 3 plus grands odds ratios:
           Variable  Coefficient  Odds ratio
18  WindGustDir_cos     0.369622    1.447188
6       Humidity3pm     0.338239    1.402476
22   WindDir3pm_cos     0.310061    1.363509


In [15]:
# CELL 15: J+1 : valeurs SHAP du modèle XGBoost pour chaque jour du test (1.f)

# TreeExplainer : valeurs SHAP exactes pour les arbres, en log-odds (marge du modèle) ; une colonne par variable,
# puis la valeur de base (marge moyenne du modèle), identique pour toutes les lignes
explainer_J1 = shap.TreeExplainer(modeles_J1['XGBoost'])
valeurs_shap_J1 = explainer_J1.shap_values(X_test_arbres_J1)
shap_J1 = pd.DataFrame(valeurs_shap_J1, columns=X_test_arbres_J1.columns)
shap_J1['Valeur de base'] = explainer_J1.expected_value
shap_J1.to_csv(dossier_app + 'shap_xgboost_J1.csv', index=False)

# Contrôle : valeur de base + somme des valeurs SHAP = marge ; probabilité = 1 / (1 + exp(-marge))
marge_J1 = shap_J1['Valeur de base'] + valeurs_shap_J1.sum(axis=1)
ecart_J1 = np.abs(1 / (1 + np.exp(-marge_J1)) - probas_J1['XGBoost']).max()
print(f"shap_xgboost_J1.csv: {shap_J1.shape[0]} lignes × {shap_J1.shape[1]} colonnes | "
      f"valeur de base = {explainer_J1.expected_value:.4f}")
print(f"Écart maximal entre la probabilité reconstruite par SHAP et la probabilité du modèle: {ecart_J1:.2e}")

shap_xgboost_J1.csv: 28445 lignes × 25 colonnes | valeur de base = 0.0090
Écart maximal entre la probabilité reconstruite par SHAP et la probabilité du modèle: 7.75e-07


In [16]:
# CELL 16: J+2 : valeurs SHAP du modèle XGBoost pour chaque jour du test (1.f)

# TreeExplainer : valeurs SHAP exactes pour les arbres, en log-odds (marge du modèle) ; une colonne par variable,
# puis la valeur de base (marge moyenne du modèle), identique pour toutes les lignes
explainer_J2 = shap.TreeExplainer(modeles_J2['XGBoost'])
valeurs_shap_J2 = explainer_J2.shap_values(X_test_arbres_J2)
shap_J2 = pd.DataFrame(valeurs_shap_J2, columns=X_test_arbres_J2.columns)
shap_J2['Valeur de base'] = explainer_J2.expected_value
shap_J2.to_csv(dossier_app + 'shap_xgboost_J2.csv', index=False)

# Contrôle : valeur de base + somme des valeurs SHAP = marge ; probabilité = 1 / (1 + exp(-marge))
marge_J2 = shap_J2['Valeur de base'] + valeurs_shap_J2.sum(axis=1)
ecart_J2 = np.abs(1 / (1 + np.exp(-marge_J2)) - probas_J2['XGBoost']).max()
print(f"shap_xgboost_J2.csv: {shap_J2.shape[0]} lignes × {shap_J2.shape[1]} colonnes | "
      f"valeur de base = {explainer_J2.expected_value:.4f}")
print(f"Écart maximal entre la probabilité reconstruite par SHAP et la probabilité du modèle: {ecart_J2:.2e}")

shap_xgboost_J2.csv: 28078 lignes × 25 colonnes | valeur de base = 0.0017
Écart maximal entre la probabilité reconstruite par SHAP et la probabilité du modèle: 1.79e-07


In [17]:
# CELL 17: Fonction des scores par station (1.g)

def scores_stations(probas, seuils):
    # Une ligne par modèle et par station : jours de test, jours de pluie, F1, précision et rappel au seuil retenu
    # (scores_seuil de graphiques.py : précision fixée à 0 si aucune prédiction positive, règle de 03 CELL 46)
    lignes = []
    for nom in MODELES:
        for station in np.sort(probas['Location'].unique()):
            p = probas[probas['Location'] == station]
            matrice, precision, rappel, f1 = scores_seuil(p['Cible'], p[nom], seuils[nom])
            lignes.append({'Modèle': nom, 'Station': station, 'Jours de test': len(p),
                           'Jours de pluie': int(p['Cible'].sum()), 'F1': f1, 'Précision': precision, 'Rappel': rappel})
    return pd.DataFrame(lignes)

In [18]:
# CELL 18: Scores par station et par modèle, J+1 et J+2 (1.g)

stations_J1 = scores_stations(probas_J1, parametres_J1['seuil_retenu'])
stations_J2 = scores_stations(probas_J2, parametres_J2['seuil_retenu'])
stations_J1.to_csv(dossier_app + 'stations_J1.csv', index=False)
stations_J2.to_csv(dossier_app + 'stations_J2.csv', index=False)

# Résumé à comparer aux sorties de 03 (CELL 74 pour J+1, CELL 133 pour J+2)
for nom in MODELES:
    r1 = stations_J1[stations_J1['Modèle'] == nom]
    r2 = stations_J2[stations_J2['Modèle'] == nom]
    print(f"{nom} | J+1: F1 moyen des stations = {r1['F1'].mean():.4f}, minimum = {r1['F1'].min():.4f} | "
          f"J+2: F1 moyen des stations = {r2['F1'].mean():.4f}, minimum = {r2['F1'].min():.4f}")

Régression logistique | J+1: F1 moyen des stations = 0.6088, minimum = 0.4174 | J+2: F1 moyen des stations = 0.4223, minimum = 0.0909
Random Forest | J+1: F1 moyen des stations = 0.6344, minimum = 0.4138 | J+2: F1 moyen des stations = 0.4430, minimum = 0.1923
XGBoost | J+1: F1 moyen des stations = 0.6516, minimum = 0.4195 | J+2: F1 moyen des stations = 0.4101, minimum = 0.1395
KNN | J+1: F1 moyen des stations = 0.5600, minimum = 0.3833 | J+2: F1 moyen des stations = 0.4069, minimum = 0.1788
Réseau de neurones | J+1: F1 moyen des stations = 0.6419, minimum = 0.4207 | J+2: F1 moyen des stations = 0.4492, minimum = 0.1750


In [19]:
# CELL 19: Pré-traitement : pourcentage de valeurs manquantes par variable (1.h)

# Sortie de 02_preprocessing, CELL 2 (données brutes, 2 décimales)
pretraitement_manquants = pd.DataFrame({
    'Variable': ['Sunshine', 'Evaporation', 'Cloud3pm', 'Cloud9am', 'Pressure9am', 'Pressure3pm', 'WindDir9am', 'WindGustDir', 'WindGustSpeed', 'Humidity3pm', 'WindDir3pm', 'Temp3pm']
    + ['RainTomorrow', 'Rainfall', 'RainToday', 'WindSpeed3pm', 'Humidity9am', 'WindSpeed9am', 'Temp9am', 'MinTemp', 'MaxTemp', 'Date', 'Location'],
    'Pourcentage': [48.01, 43.17, 40.81, 38.42, 10.36, 10.33, 7.26, 7.1, 7.06, 3.1, 2.91, 2.48]
    + [2.25, 2.24, 2.24, 2.11, 1.82, 1.21, 1.21, 1.02, 0.87, 0.0, 0.0]
})
pretraitement_manquants.to_csv(dossier_app + 'pretraitement_valeurs_manquantes.csv', index=False)
print(pretraitement_manquants)

         Variable  Pourcentage
0        Sunshine        48.01
1     Evaporation        43.17
2        Cloud3pm        40.81
3        Cloud9am        38.42
4     Pressure9am        10.36
5     Pressure3pm        10.33
6      WindDir9am         7.26
7     WindGustDir         7.10
8   WindGustSpeed         7.06
9     Humidity3pm         3.10
10     WindDir3pm         2.91
11        Temp3pm         2.48
12   RainTomorrow         2.25
13       Rainfall         2.24
14      RainToday         2.24
15   WindSpeed3pm         2.11
16    Humidity9am         1.82
17   WindSpeed9am         1.21
18        Temp9am         1.21
19        MinTemp         1.02
20        MaxTemp         0.87
21           Date         0.00
22       Location         0.00


In [20]:
# CELL 20: Pré-traitement : comparaison de l'imputation globale et de l'imputation locale (1.h)

# Sortie de 02_preprocessing, CELL 33 : F1 moyen sur les 5 plis de chaque modèle (pleine précision, cible J+1)
pretraitement_imputation = pd.DataFrame({
    'Modèle': ['Régression logistique', 'Random Forest', 'XGBoost', 'KNN', 'Réseau de neurones'],
    'Imputation globale': [0.5750611120663919, 0.5866452709554814, 0.6123873092854876, 0.5326061497366065, 0.6113842643852966],
    'Imputation locale': [0.5762688140457385, 0.586680652906734, 0.6096036522888321, 0.5335043876761014, 0.6110754340075133]
})
pretraitement_imputation.to_csv(dossier_app + 'pretraitement_imputation_globale_locale.csv', index=False)
print(pretraitement_imputation)

                  Modèle  Imputation globale  Imputation locale
0  Régression logistique            0.575061           0.576269
1          Random Forest            0.586645           0.586681
2                XGBoost            0.612387           0.609604
3                    KNN            0.532606           0.533504
4     Réseau de neurones            0.611384           0.611075


In [21]:
# CELL 21: Pré-traitement : biais évité par l'imputation selon le lien entre les mesures de 9 h et de 15 h (1.h)

# Sorties de 02_preprocessing : CELL 15 (corrélation de Pearson 9 h / 15 h, 4 décimales) et CELL 17 (bornes IQR des
# écarts mesurés, lignes avec une seule valeur manquante, pourcentages d'écarts hors bornes, 2 décimales)
lignes = [
    ['Température 9 h / 15 h', 'Temp9am / Temp3pm', 0.8598, 1265, -5.55, 14.85, 6.64, 0.56],
    ['Humidité 9 h / 15 h', 'Humidity9am / Humidity3pm', 0.6649, 1497, -61.0, 27.0, 1.67, 1.01],
    ['Pression 9 h / 15 h', 'Pressure9am / Pressure3pm', 0.9619, 387, -7.3, 2.3, 48.58, 2.66]
]
pretraitement_biais = pd.DataFrame(lignes, columns=['Paire', 'Variables', 'Corrélation 9 h / 15 h',
                                                    'Lignes avec une valeur manquante', 'Borne basse', 'Borne haute',
                                                    'Écarts hypothétiques hors bornes (%)',
                                                    'Écarts mesurés hors bornes (%)'])
pretraitement_biais.to_csv(dossier_app + 'pretraitement_biais_evite.csv', index=False)
print(pretraitement_biais)

                    Paire                  Variables  Corrélation 9 h / 15 h  Lignes avec une valeur manquante  Borne basse  Borne haute  Écarts hypothétiques hors bornes (%)  \
0  Température 9 h / 15 h          Temp9am / Temp3pm                  0.8598                              1265        -5.55        14.85                                  6.64   
1     Humidité 9 h / 15 h  Humidity9am / Humidity3pm                  0.6649                              1497       -61.00        27.00                                  1.67   
2     Pression 9 h / 15 h  Pressure9am / Pressure3pm                  0.9619                               387        -7.30         2.30                                 48.58   

   Écarts mesurés hors bornes (%)  
0                            0.56  
1                            1.01  
2                            2.66  


In [22]:
# CELL 22: Pré-traitement : paires de variables redondantes et décisions du filtre (1.h)

# Sorties de 02_preprocessing : CELL 56 (paires avec |r| >= 0,8 et |corrélation avec la cible| de chaque variable,
# 4 décimales) et CELL 57 (décision : on retire la variable la moins corrélée à la cible)
lignes = [
    ['MaxTemp', 'Temp3pm', 0.9831, 0.9831, 0.1594, 0.1924, 'MaxTemp retirée (Temp3pm conservée)'],
    ['Pressure9am', 'Pressure3pm', 0.962, 0.962, 0.2316, 0.2111, 'Pressure3pm retirée (Pressure9am conservée)'],
    ['MinTemp', 'Temp9am', 0.8994, 0.8994, 0.087, 0.0275, 'Temp9am retirée (MinTemp conservée)'],
    ['MaxTemp', 'Temp9am', 0.8863, 0.8863, 0.1594, 0.0275, 'Paire ignorée (MaxTemp déjà retirée)'],
    ['Temp9am', 'Temp3pm', 0.8628, 0.8628, 0.0275, 0.1924, 'Paire ignorée (Temp9am déjà retirée)']
]
pretraitement_paires = pd.DataFrame(lignes, columns=['Variable 1', 'Variable 2', 'r', '|r|', '|Corrélation cible| 1',
                                                     '|Corrélation cible| 2', 'Décision'])
pretraitement_paires.to_csv(dossier_app + 'pretraitement_paires_redondantes.csv', index=False)
print(pretraitement_paires)

    Variable 1   Variable 2       r     |r|  |Corrélation cible| 1  |Corrélation cible| 2                                     Décision
0      MaxTemp      Temp3pm  0.9831  0.9831                 0.1594                 0.1924          MaxTemp retirée (Temp3pm conservée)
1  Pressure9am  Pressure3pm  0.9620  0.9620                 0.2316                 0.2111  Pressure3pm retirée (Pressure9am conservée)
2      MinTemp      Temp9am  0.8994  0.8994                 0.0870                 0.0275          Temp9am retirée (MinTemp conservée)
3      MaxTemp      Temp9am  0.8863  0.8863                 0.1594                 0.0275         Paire ignorée (MaxTemp déjà retirée)
4      Temp9am      Temp3pm  0.8628  0.8628                 0.0275                 0.1924         Paire ignorée (Temp9am déjà retirée)


In [23]:
# CELL 23: Pré-traitement : comparaison du jeu complet et du jeu réduit (1.h)

# Sortie de 02_preprocessing, CELL 69 : F1 moyen sur les 5 plis de chaque modèle (pleine précision, cible J+1)
pretraitement_jeux = pd.DataFrame({
    'Modèle': ['Régression logistique', 'Random Forest', 'XGBoost', 'KNN', 'Réseau de neurones'],
    'Jeu complet': [0.5757729313194723, 0.584922769719568, 0.6079673445131231, 0.5353269230766278, 0.6096291686958116],
    'Jeu réduit': [0.5769049074138771, 0.5835617297617585, 0.6071386617314817, 0.540485654871149, 0.6099990717299538]
})
pretraitement_jeux.to_csv(dossier_app + 'pretraitement_jeu_complet_reduit.csv', index=False)
print(pretraitement_jeux)

                  Modèle  Jeu complet  Jeu réduit
0  Régression logistique     0.575773    0.576905
1          Random Forest     0.584923    0.583562
2                XGBoost     0.607967    0.607139
3                    KNN     0.535327    0.540486
4     Réseau de neurones     0.609629    0.609999


In [24]:
# CELL 24: Pré-traitement : choix du scaler de chaque colonne (1.h)

# Sortie de 02_preprocessing, CELL 75 (skewness à 3 décimales, pourcentage de valeurs aberrantes IQR à 2 décimales) ;
# dans 02, chaque scaler correspond à un seul motif de la règle de décision
lignes = [
    ['Month_cos', 0.052, 0.0, 'MinMaxScaler'], ['Month_sin', 0.056, 0.0, 'MinMaxScaler'],
    ['Season_cos', 0.055, 0.0, 'MinMaxScaler'], ['Season_sin', 0.009, 0.0, 'MinMaxScaler'],
    ['WindDir3pm_cos', 0.111, 0.0, 'MinMaxScaler'], ['WindDir3pm_sin', -0.0, 0.0, 'MinMaxScaler'],
    ['WindDir9am_cos', -0.025, 0.0, 'MinMaxScaler'], ['WindDir9am_sin', 0.019, 0.0, 'MinMaxScaler'],
    ['WindGustDir_cos', 0.097, 0.0, 'MinMaxScaler'], ['WindGustDir_sin', 0.119, 0.0, 'MinMaxScaler'],
    ['Humidity9am', -0.489, 1.06, 'RobustScaler'], ['Humidity_diff', 0.07, 1.01, 'RobustScaler'],
    ['Pressure_diff', 0.637, 2.66, 'RobustScaler'], ['Rainfall', 10.173, 17.93, 'RobustScaler'],
    ['WindGustSpeed', 0.91, 2.29, 'RobustScaler'], ['WindSpeed3pm', 0.62, 1.77, 'RobustScaler'],
    ['WindSpeed9am', 0.769, 1.28, 'RobustScaler'], ['Humidity3pm', 0.014, 0.0, 'StandardScaler'],
    ['Location_encoded', -0.352, 0.82, 'StandardScaler'], ['Location_encoded_J2', -0.283, 0.82, 'StandardScaler'],
    ['MinTemp', 0.03, 0.04, 'StandardScaler'], ['Pressure9am', -0.097, 0.87, 'StandardScaler'],
    ['Temp3pm', 0.256, 0.61, 'StandardScaler'], ['Temp_diff', 0.308, 0.56, 'StandardScaler']
]
pretraitement_scalers = pd.DataFrame(lignes, columns=['Variable', 'Skewness', 'Valeurs aberrantes (%)', 'Scaler'])
motifs = {'MinMaxScaler': "distribution en U non normale d'un sinus / cosinus, sans valeurs aberrantes",
          'RobustScaler': "au moins 1 % de valeurs aberrantes",
          'StandardScaler': "quasi-normale (|skew| < 0,5), moins de 1 % de valeurs aberrantes"}
pretraitement_scalers['Motif'] = pretraitement_scalers['Scaler'].map(motifs)
pretraitement_scalers.to_csv(dossier_app + 'pretraitement_scalers.csv', index=False)
print(pretraitement_scalers)

               Variable  Skewness  Valeurs aberrantes (%)          Scaler                                              Motif
0             Month_cos     0.052                    0.00    MinMaxScaler  distribution en U non normale d'un sinus / cos...
1             Month_sin     0.056                    0.00    MinMaxScaler  distribution en U non normale d'un sinus / cos...
2            Season_cos     0.055                    0.00    MinMaxScaler  distribution en U non normale d'un sinus / cos...
3            Season_sin     0.009                    0.00    MinMaxScaler  distribution en U non normale d'un sinus / cos...
4        WindDir3pm_cos     0.111                    0.00    MinMaxScaler  distribution en U non normale d'un sinus / cos...
5        WindDir3pm_sin    -0.000                    0.00    MinMaxScaler  distribution en U non normale d'un sinus / cos...
6        WindDir9am_cos    -0.025                    0.00    MinMaxScaler  distribution en U non normale d'un sinus / cos...


In [25]:
# CELL 25: Pré-traitement : corrélation des nouvelles variables avec la cible (1.h)

# Sortie de 02_preprocessing, CELL 47 (X_train, cible J+1, 4 décimales)
pretraitement_correlations = pd.DataFrame({
    'Variable': ['Temp_diff', 'Humidity_diff', 'Pressure_diff', 'Season', 'Month'],
    'Pearson': [-0.3259, 0.2631, 0.0824, 0.0133, 0.0096],
    'Spearman': [-0.3332, 0.2561, 0.0848, 0.0127, 0.0087]
})
pretraitement_correlations.to_csv(dossier_app + 'pretraitement_correlations_nouvelles_variables.csv', index=False)
print(pretraitement_correlations)

        Variable  Pearson  Spearman
0      Temp_diff  -0.3259   -0.3332
1  Humidity_diff   0.2631    0.2561
2  Pressure_diff   0.0824    0.0848
3         Season   0.0133    0.0127
4          Month   0.0096    0.0087


In [26]:
# CELL 26: Pré-traitement : position des mois et des directions du vent sur le cercle (encodage cyclique)

# Mêmes formules que 02_preprocessing : sin / cos de 2π × numéro / 12 (mois) ou / 16 (directions, numéros des artefacts)
noms_mois = ['Janvier', 'Février', 'Mars', 'Avril', 'Mai', 'Juin', 'Juillet', 'Août', 'Septembre', 'Octobre',
             'Novembre', 'Décembre']
lignes = []
for mois in range(1, 13):
    lignes.append(['Mois', noms_mois[mois - 1], str(mois), mois, np.sin(2 * np.pi * mois / 12),
                   np.cos(2 * np.pi * mois / 12)])
# Directions : libellé français (W, ouest -> O), code d'origine conservé
for code in artefacts['numero_direction']:
    numero = artefacts['numero_direction'][code]
    lignes.append(['Direction du vent', code.replace('W', 'O'), code, numero, np.sin(2 * np.pi * numero / 16),
                   np.cos(2 * np.pi * numero / 16)])
encodage_cyclique = pd.DataFrame(lignes, columns=['Type', 'Libellé', 'Code', 'Numéro', 'sin', 'cos'])
encodage_cyclique.to_csv(dossier_app + 'pretraitement_encodage_cyclique.csv', index=False)
print(encodage_cyclique)

                 Type    Libellé Code  Numéro           sin           cos
0                Mois    Janvier    1       1  5.000000e-01  8.660254e-01
1                Mois    Février    2       2  8.660254e-01  5.000000e-01
2                Mois       Mars    3       3  1.000000e+00  6.123234e-17
3                Mois      Avril    4       4  8.660254e-01 -5.000000e-01
4                Mois        Mai    5       5  5.000000e-01 -8.660254e-01
5                Mois       Juin    6       6  1.224647e-16 -1.000000e+00
6                Mois    Juillet    7       7 -5.000000e-01 -8.660254e-01
7                Mois       Août    8       8 -8.660254e-01 -5.000000e-01
8                Mois  Septembre    9       9 -1.000000e+00 -1.836970e-16
9                Mois    Octobre   10      10 -8.660254e-01  5.000000e-01
10               Mois   Novembre   11      11 -5.000000e-01  8.660254e-01
11               Mois   Décembre   12      12 -2.449294e-16  1.000000e+00
12  Direction du vent          N    N 

In [27]:
# CELL 27: Exploration : répartition de la cible (1.i)

# Cible numérique : Yes = 1, No = 0, valeurs manquantes conservées (exclues des comptes et des moyennes)
brut['Cible'] = brut['RainTomorrow'].map({'No': 0, 'Yes': 1})
effectifs = brut['RainTomorrow'].value_counts()
exploration_cible = pd.DataFrame({
    'Classe': ['Pas de pluie', 'Pluie'],
    'Nombre de jours': [effectifs['No'], effectifs['Yes']],
    'Pourcentage': [effectifs['No'] / effectifs.sum() * 100, effectifs['Yes'] / effectifs.sum() * 100]
})
exploration_cible.to_csv(dossier_app + 'exploration_repartition_cible.csv', index=False)
print(exploration_cible)

         Classe  Nombre de jours  Pourcentage
0  Pas de pluie           110316    77.581878
1         Pluie            31877    22.418122


In [28]:
# CELL 28: Exploration : pourcentage de valeurs manquantes par variable (1.i)

# Variables des données brutes ayant au moins une valeur manquante (colonne Cible ajoutée en CELL 27 exclue)
colonnes_brutes = brut.drop(columns=['Cible']).columns
exploration_manquants = pd.DataFrame({
    'Variable': colonnes_brutes,
    'Nombre': brut[colonnes_brutes].isnull().sum().values,
    'Pourcentage': brut[colonnes_brutes].isnull().sum().values / len(brut) * 100
})
exploration_manquants = exploration_manquants[exploration_manquants['Nombre'] > 0]
exploration_manquants = exploration_manquants.sort_values('Pourcentage', ascending=False)
exploration_manquants.to_csv(dossier_app + 'exploration_valeurs_manquantes.csv', index=False)
print(exploration_manquants)

         Variable  Nombre  Pourcentage
6        Sunshine   69835    48.009762
5     Evaporation   62790    43.166506
18       Cloud3pm   59358    40.807095
17       Cloud9am   55888    38.421559
15    Pressure9am   15065    10.356799
16    Pressure3pm   15028    10.331363
9      WindDir9am   10566     7.263853
7     WindGustDir   10326     7.098859
8   WindGustSpeed   10263     7.055548
14    Humidity3pm    4507     3.098446
10     WindDir3pm    4228     2.906641
20        Temp3pm    3609     2.481094
22   RainTomorrow    3267     2.245978
4        Rainfall    3261     2.241853
21      RainToday    3261     2.241853
12   WindSpeed3pm    3062     2.105046
13    Humidity9am    2654     1.824557
11   WindSpeed9am    1767     1.214767
19        Temp9am    1767     1.214767
2         MinTemp    1485     1.020899
3         MaxTemp    1261     0.866905


In [29]:
# CELL 29: Exploration : corrélation de chaque variable numérique avec la cible (1.i)

# Variables numériques des données brutes (float64, comme 01_exploration) ; corrélation de Pearson avec la cible 0 / 1
colonnes_numeriques = brut.drop(columns=['Cible']).select_dtypes(include=['float64']).columns
correlations = []
for col in colonnes_numeriques:
    correlations.append(brut['Cible'].corr(brut[col]))
exploration_correlations = pd.DataFrame({'Variable': colonnes_numeriques, 'Corrélation': correlations})
exploration_correlations = exploration_correlations.sort_values('Corrélation', ascending=False)
exploration_correlations.to_csv(dossier_app + 'exploration_correlations_cible.csv', index=False)
print(exploration_correlations)

         Variable  Corrélation
9     Humidity3pm     0.446160
13       Cloud3pm     0.381870
12       Cloud9am     0.317380
8     Humidity9am     0.257161
2        Rainfall     0.239032
5   WindGustSpeed     0.234010
6    WindSpeed9am     0.090995
7    WindSpeed3pm     0.087817
0         MinTemp     0.083936
14        Temp9am    -0.025691
3     Evaporation    -0.119285
1         MaxTemp    -0.159237
15        Temp3pm    -0.192424
11    Pressure3pm    -0.226031
10    Pressure9am    -0.246371
4        Sunshine    -0.450768


In [30]:
# CELL 30: Exploration : taux de pluie le lendemain par station (1.i)

# Pourcentage de jours avec RainTomorrow = Yes (cible connue) et nombre de jours avec la cible connue, par station
taux_station = brut.groupby('Location')['Cible'].mean() * 100
jours_station = brut.groupby('Location')['Cible'].count()
exploration_stations = pd.DataFrame({'Station': taux_station.index, 'Taux de pluie (%)': taux_station.values,
                                     'Jours': jours_station.values})
exploration_stations = exploration_stations.sort_values('Taux de pluie (%)', ascending=False)
exploration_stations.to_csv(dossier_app + 'exploration_pluie_par_station.csv', index=False)
print(exploration_stations.head())
print(exploration_stations.tail())

          Station  Taux de pluie (%)  Jours
33       Portland          36.548732   2996
43        Walpole          33.664420   2819
8          Cairns          31.793842   2988
12       Dartmoor          31.328576   2943
27  NorfolkIsland          31.005398   2964
         Station  Taux de pluie (%)  Jours
10         Cobar          12.918340   2988
20       Mildura          10.874626   3007
3   AliceSprings           8.050148   3031
41         Uluru           7.626561   1521
48       Woomera           6.755853   2990


In [31]:
# CELL 31: Exploration : taux de pluie le lendemain par mois sur toute la période (1.i)

# Taux de RainTomorrow = Yes par année et par mois, toutes stations confondues ; le mois est daté de son premier jour
dates = pd.to_datetime(brut['Date'])
brut['Année'] = dates.dt.year
brut['Mois'] = dates.dt.month
taux_mois = brut.groupby(['Année', 'Mois'])['Cible'].mean() * 100
jours_mois = brut.groupby(['Année', 'Mois'])['Cible'].count()
premiers_jours = []
for i in range(len(taux_mois)):
    premiers_jours.append(f"{taux_mois.index[i][0]}-{taux_mois.index[i][1]:02d}-01")
exploration_mensuel = pd.DataFrame({'Mois': pd.to_datetime(premiers_jours), 'Taux de pluie (%)': taux_mois.values,
                                    'Jours': jours_mois.values})
exploration_mensuel.to_csv(dossier_app + 'exploration_taux_pluie_mensuel.csv', index=False)
print(f"{len(exploration_mensuel)} mois, du {str(exploration_mensuel['Mois'].min())[:7]} "
      f"au {str(exploration_mensuel['Mois'].max())[:7]}")
print(exploration_mensuel.head())

113 mois, du 2007-11 au 2017-06
        Mois  Taux de pluie (%)  Jours
0 2007-11-01          26.666667     30
1 2007-12-01          35.483871     31
2 2008-01-01          19.354839     31
3 2008-02-01          37.931034     58
4 2008-03-01          12.903226     62


In [32]:
# CELL 32: Plage observée de chaque variable numérique brute (analyse de sensibilité)

# Minimum et maximum de chaque variable numérique sur toutes les données brutes (valeurs manquantes exclues)
exploration_plages = pd.DataFrame({'Variable': colonnes_numeriques, 'Minimum': brut[colonnes_numeriques].min().values,
                                   'Maximum': brut[colonnes_numeriques].max().values})
exploration_plages.to_csv(dossier_app + 'plages_variables.csv', index=False)
print(exploration_plages)

         Variable  Minimum  Maximum
0         MinTemp     -8.5     33.9
1         MaxTemp     -4.8     48.1
2        Rainfall      0.0    371.0
3     Evaporation      0.0    145.0
4        Sunshine      0.0     14.5
5   WindGustSpeed      6.0    135.0
6    WindSpeed9am      0.0    130.0
7    WindSpeed3pm      0.0     87.0
8     Humidity9am      0.0    100.0
9     Humidity3pm      0.0    100.0
10    Pressure9am    980.5   1041.0
11    Pressure3pm    977.1   1039.6
12       Cloud9am      0.0      9.0
13       Cloud3pm      0.0      9.0
14        Temp9am     -7.2     40.2
15        Temp3pm     -5.4     46.7


In [33]:
# CELL 33: Liste des fichiers exportés dans data/output/app/ et leurs dimensions

fichiers_exportes = ['scores_plis_J1.csv', 'scores_plis_J2.csv', 'progression_J1.csv', 'progression_J2.csv',
                     'resultats_test_J1.csv', 'resultats_test_J2.csv', 'probas_test_J1.csv', 'probas_test_J2.csv',
                     'importances_xgboost_J1.csv', 'importances_xgboost_J2.csv', 'odds_ratios_J1.csv',
                     'odds_ratios_J2.csv', 'shap_xgboost_J1.csv', 'shap_xgboost_J2.csv', 'stations_J1.csv',
                     'stations_J2.csv', 'pretraitement_valeurs_manquantes.csv',
                     'pretraitement_imputation_globale_locale.csv', 'pretraitement_biais_evite.csv',
                     'pretraitement_paires_redondantes.csv', 'pretraitement_jeu_complet_reduit.csv',
                     'pretraitement_scalers.csv', 'pretraitement_correlations_nouvelles_variables.csv',
                     'pretraitement_encodage_cyclique.csv', 'exploration_repartition_cible.csv',
                     'exploration_valeurs_manquantes.csv', 'exploration_correlations_cible.csv',
                     'exploration_pluie_par_station.csv', 'exploration_taux_pluie_mensuel.csv', 'plages_variables.csv']
print(f"Fichiers exportés dans data/output/app/ ({len(fichiers_exportes)}):")
for nom_fichier in fichiers_exportes:
    relu = pd.read_csv(dossier_app + nom_fichier)
    print(f"  {nom_fichier}: {relu.shape[0]} lignes × {relu.shape[1]} colonnes")

Fichiers exportés dans data/output/app/ (30):
  scores_plis_J1.csv: 5 lignes × 7 colonnes
  scores_plis_J2.csv: 5 lignes × 7 colonnes
  progression_J1.csv: 5 lignes × 6 colonnes
  progression_J2.csv: 5 lignes × 6 colonnes
  resultats_test_J1.csv: 6 lignes × 10 colonnes
  resultats_test_J2.csv: 6 lignes × 10 colonnes


  probas_test_J1.csv: 28445 lignes × 8 colonnes


  probas_test_J2.csv: 28078 lignes × 8 colonnes
  importances_xgboost_J1.csv: 24 lignes × 2 colonnes


  importances_xgboost_J2.csv: 24 lignes × 2 colonnes
  odds_ratios_J1.csv: 24 lignes × 3 colonnes
  odds_ratios_J2.csv: 24 lignes × 3 colonnes


  shap_xgboost_J1.csv: 28445 lignes × 25 colonnes


  shap_xgboost_J2.csv: 28078 lignes × 25 colonnes
  stations_J1.csv: 245 lignes × 7 colonnes
  stations_J2.csv: 245 lignes × 7 colonnes
  pretraitement_valeurs_manquantes.csv: 23 lignes × 2 colonnes
  pretraitement_imputation_globale_locale.csv: 5 lignes × 3 colonnes
  pretraitement_biais_evite.csv: 3 lignes × 8 colonnes
  pretraitement_paires_redondantes.csv: 5 lignes × 7 colonnes
  pretraitement_jeu_complet_reduit.csv: 5 lignes × 3 colonnes
  pretraitement_scalers.csv: 24 lignes × 5 colonnes
  pretraitement_correlations_nouvelles_variables.csv: 5 lignes × 3 colonnes
  pretraitement_encodage_cyclique.csv: 28 lignes × 6 colonnes
  exploration_repartition_cible.csv: 2 lignes × 3 colonnes
  exploration_valeurs_manquantes.csv: 21 lignes × 3 colonnes
  exploration_correlations_cible.csv: 16 lignes × 2 colonnes
  exploration_pluie_par_station.csv: 49 lignes × 3 colonnes
  exploration_taux_pluie_mensuel.csv: 113 lignes × 3 colonnes
  plages_variables.csv: 16 lignes × 3 colonnes


In [34]:
# CELL 34: Relecture des fichiers exportés (les graphiques utilisent les mêmes fichiers que l'application)

app = {}
for nom_fichier in fichiers_exportes:
    app[nom_fichier] = pd.read_csv(dossier_app + nom_fichier)
app['exploration_taux_pluie_mensuel.csv']['Mois'] = pd.to_datetime(app['exploration_taux_pluie_mensuel.csv']['Mois'])
print(f"{len(app)} fichiers relus")

30 fichiers relus


In [35]:
# CELL 35: Exploration : répartition de la cible

fig = graphique_repartition_cible(app['exploration_repartition_cible.csv'])
fig.write_image(dossier_figures + 'graphique_repartition_cible.png', width=1200)
fig.show()

In [36]:
# CELL 36: Exploration : valeurs manquantes par variable (seuil de 20 %)

fig = graphique_valeurs_manquantes(app['exploration_valeurs_manquantes.csv'])
fig.write_image(dossier_figures + 'graphique_valeurs_manquantes.png', width=1200)
fig.show()

In [37]:
# CELL 37: Exploration : distribution d'une variable selon la pluie (exemple : Humidity3pm)

fig = graphique_distribution_pluie(brut, 'Humidity3pm')
fig.write_image(dossier_figures + 'graphique_distribution_pluie.png', width=1200)
fig.show()

In [38]:
# CELL 38: Exploration : corrélation des variables avec la cible

fig = graphique_correlation_cible(app['exploration_correlations_cible.csv'])
fig.write_image(dossier_figures + 'graphique_correlation_cible.png', width=1200)
fig.show()

In [39]:
# CELL 39: Exploration : fréquence de la pluie par station

fig = graphique_pluie_par_station(app['exploration_pluie_par_station.csv'])
fig.write_image(dossier_figures + 'graphique_pluie_par_station.png', width=1200)
fig.show()

In [40]:
# CELL 40: Pré-traitement : taux de pluie mensuel et coupure entraînement / test

fig = graphique_taux_mensuel(app['exploration_taux_pluie_mensuel.csv'], artefacts['date_coupure'])
fig.write_image(dossier_figures + 'graphique_taux_mensuel.png', width=1200)
fig.show()

In [41]:
# CELL 41: Pré-traitement : biais évité par l'imputation selon le lien

fig = graphique_biais_evite(app['pretraitement_biais_evite.csv'])
fig.write_image(dossier_figures + 'graphique_biais_evite.png', width=1200)
fig.show()

In [42]:
# CELL 42: Pré-traitement : imputation globale et imputation locale (5 modèles)

fig = graphique_imputation(app['pretraitement_imputation_globale_locale.csv'])
fig.write_image(dossier_figures + 'graphique_imputation.png', width=1200)
fig.show()

In [43]:
# CELL 43: Pré-traitement : cercle de l'encodage cyclique (exemple : directions du vent)

fig = graphique_encodage_cyclique(app['pretraitement_encodage_cyclique.csv'], 'Direction du vent')
fig.write_image(dossier_figures + 'graphique_encodage_cyclique.png', width=1200)
fig.show()

In [44]:
# CELL 44: Pré-traitement : paires redondantes

fig = graphique_paires_redondantes(app['pretraitement_paires_redondantes.csv'])
fig.write_image(dossier_figures + 'graphique_paires_redondantes.png', width=1200)
fig.show()

In [45]:
# CELL 45: Pré-traitement : jeu complet et jeu réduit (5 modèles)

fig = graphique_jeu_reduit(app['pretraitement_jeu_complet_reduit.csv'])
fig.write_image(dossier_figures + 'graphique_jeu_reduit.png', width=1200)
fig.show()

In [46]:
# CELL 46: Modélisation : F1 par pli (exemple : J+1)

fig = graphique_f1_par_pli(app['scores_plis_J1.csv'], 'J+1')
fig.write_image(dossier_figures + 'graphique_f1_par_pli.png', width=1200)
fig.show()

In [47]:
# CELL 47: Modélisation : évolution des performances (exemple : XGBoost, J+1)

fig = graphique_evolution(app['progression_J1.csv'], 'XGBoost', 'J+1')
fig.write_image(dossier_figures + 'graphique_evolution.png', width=1200)
fig.show()

In [48]:
# CELL 48: Modélisation : comparaison des modèles (exemple : F1, J+1)

fig = graphique_comparaison_modeles(app['resultats_test_J1.csv'], 'F1', 'J+1')
fig.write_image(dossier_figures + 'graphique_comparaison_modeles.png', width=1200)
fig.show()

In [49]:
# CELL 49: Modélisation : seuil de décision (exemple : XGBoost au seuil retenu, J+1)

fig = graphique_seuil(app['probas_test_J1.csv'], 'XGBoost', parametres_J1['seuil_retenu']['XGBoost'], 'J+1')
fig.write_image(dossier_figures + 'graphique_seuil.png', width=1200)
fig.show()

In [50]:
# CELL 50: Modélisation : importance des variables (XGBoost, exemple : J+1)

fig = graphique_importance(app['importances_xgboost_J1.csv'], 'J+1')
fig.write_image(dossier_figures + 'graphique_importance.png', width=1200)
fig.show()

In [51]:
# CELL 51: Modélisation : synthèse SHAP (XGBoost, exemple : J+1)

fig = graphique_shap_synthese(app['shap_xgboost_J1.csv'], 'J+1')
fig.write_image(dossier_figures + 'graphique_shap_synthese.png', width=1200)
fig.show()

In [52]:
# CELL 52: Modélisation : valeurs SHAP de chaque jour (XGBoost, exemple : J+1)

# Une ligne par variable (15 premières), un point par jour du test (3 000 jours tirés au hasard), couleur selon la
# valeur de la variable préparée (fichier X_test_J1_arbres.csv, mêmes lignes que shap_xgboost_J1.csv)
fig = graphique_shap_beeswarm(app['shap_xgboost_J1.csv'], X_test_arbres_J1, 'J+1')
fig.write_image(dossier_figures + 'graphique_shap_beeswarm.png', width=1200)
fig.show()

In [53]:
# CELL 53: Modélisation : odds ratios (régression logistique, exemple : J+1)

fig = graphique_odds_ratios(app['odds_ratios_J1.csv'], 'J+1')
fig.write_image(dossier_figures + 'graphique_odds_ratios.png', width=1200)
fig.show()

In [54]:
# CELL 54: Modélisation : performance par station (exemple : XGBoost, J+1)

fig = graphique_stations(app['stations_J1.csv'], 'XGBoost', 'J+1')
fig.write_image(dossier_figures + 'graphique_stations.png', width=1200)
fig.show()

In [55]:
# CELL 55: Importance intégrée de XGBoost et synthèse SHAP : 10 premières variables, J+1 et J+2

def top_10(importances, valeurs_shap):
    # 10 premières variables selon l'importance intégrée (gain, fichier déjà trié) et selon la valeur SHAP
    # absolue moyenne sur le test (mesure de référence), côte à côte
    variables = []
    moyennes = []
    for col in importances['Variable']:
        variables.append(col)
        moyennes.append(np.abs(valeurs_shap[col]).mean())
    synthese = pd.DataFrame({'Variable': variables, 'SHAP absolu moyen': moyennes})
    synthese = synthese.sort_values('SHAP absolu moyen', ascending=False)
    return pd.DataFrame({'Rang': np.arange(1, 11), 'Importance XGBoost (gain)': importances['Variable'].values[:10],
                         'Gain': importances['Importance'].values[:10], 'Synthèse SHAP': synthese['Variable'].values[:10],
                         'SHAP absolu moyen': synthese['SHAP absolu moyen'].values[:10]})


print("J+1 : 10 premières variables")
print(top_10(app['importances_xgboost_J1.csv'], app['shap_xgboost_J1.csv']))
print("\nJ+2 : 10 premières variables")
print(top_10(app['importances_xgboost_J2.csv'], app['shap_xgboost_J2.csv']))

J+1 : 10 premières variables
   Rang Importance XGBoost (gain)      Gain     Synthèse SHAP  SHAP absolu moyen
0     1               Humidity3pm  0.164236       Humidity3pm           1.108038
1     2                  Rainfall  0.093503     WindGustSpeed           0.505146
2     3                Season_sin  0.083263       Pressure9am           0.403678
3     4            WindDir3pm_cos  0.081545           Temp3pm           0.341597
4     5            WindDir9am_cos  0.081267     Pressure_diff           0.286731
5     6                Season_cos  0.078788          Rainfall           0.284238
6     7            WindDir3pm_sin  0.049996  Location_encoded           0.229775
7     8                 Month_cos  0.044782    WindDir3pm_cos           0.223542
8     9             WindGustSpeed  0.043078           MinTemp           0.208221
9    10            WindDir9am_sin  0.039322         RainToday           0.193776

J+2 : 10 premières variables
   Rang Importance XGBoost (gain)      Gain       

In [56]:
# CELL 56: Prédiction : jour du test choisi pour l'exemple (J+1, modèle de démonstration XGBoost)

# Premier jour du test à Sydney où il a plu le lendemain et où XGBoost a prédit la pluie (probabilité >= seuil retenu)
seuil_xgboost = parametres_J1['seuil_retenu']['XGBoost']
masque = (probas_J1['Location'] == 'Sydney') & (probas_J1['Cible'] == 1) & (probas_J1['XGBoost'] >= seuil_xgboost)
position = probas_J1[masque].index[0]
station = probas_J1['Location'].values[position]
date = probas_J1['Date'].values[position]
proba_exemple = probas_J1['XGBoost'].values[position]

# Ligne brute correspondante dans weatherAUS.csv (colonnes d'origine uniquement)
observation = brut[(brut['Location'] == station) & (brut['Date'] == date)].drop(columns=['Cible', 'Année', 'Mois'])
print(f"Position dans le test: {position} | {station} | {date} | cible réelle = {probas_J1['Cible'].values[position]}")
print(f"Probabilité de pluie (XGBoost) = {proba_exemple:.4f} | seuil retenu = {seuil_xgboost:.3f}")
print(observation)

Position dans le test: 170 | Sydney | 2015-11-13 | cible réelle = 1
Probabilité de pluie (XGBoost) = 0.5820 | seuil retenu = 0.302
             Date Location  MinTemp  MaxTemp  Rainfall  Evaporation  Sunshine WindGustDir  WindGustSpeed WindDir9am WindDir3pm  WindSpeed9am  WindSpeed3pm  Humidity9am  Humidity3pm  Pressure9am  \
32929  2015-11-13   Sydney     18.0     28.8       0.4          8.4       7.4         SSW           48.0          W         NE           7.0           6.0         63.0         60.0       1013.1   

       Pressure3pm  Cloud9am  Cloud3pm  Temp9am  Temp3pm RainToday RainTomorrow  
32929       1009.8       1.0       7.0     22.7     23.3        No          Yes  


In [57]:
# CELL 57: Prédiction : vérification de la préparation de la ligne brute par graphiques.py

# La ligne brute préparée par preparer_brut (artefacts de 02) doit donner la ligne du fichier de test non mis à
# l'échelle et la même probabilité que le fichier des probabilités
ligne_preparee = preparer_brut(observation, artefacts, False, artefacts['colonnes_J1'])
ecart_ligne = np.abs(ligne_preparee.values[0] - X_test_arbres_J1.values[position]).max()
proba_preparee = modeles_J1['XGBoost'].predict_proba(ligne_preparee)[:, 1][0]
print(f"Écart maximal avec la ligne de X_test_J1_arbres.csv: {ecart_ligne:.2e}")
print(f"Probabilité à partir de la ligne brute = {proba_preparee:.6f} | fichier des probabilités = {proba_exemple:.6f}")

Écart maximal avec la ligne de X_test_J1_arbres.csv: 5.55e-17
Probabilité à partir de la ligne brute = 0.581982 | fichier des probabilités = 0.581982


In [58]:
# CELL 58: Prédiction : jauge de probabilité

fig = graphique_jauge(proba_exemple, seuil_xgboost)
fig.write_image(dossier_figures + 'graphique_jauge.png', width=1200)
fig.show()

In [59]:
# CELL 59: Prédiction : contribution des variables (valeurs SHAP de la ligne choisie)

# Valeurs SHAP de la ligne (fichier shap_xgboost_J1.csv) et valeurs des variables préparées (non mises à l'échelle)
shap_ligne = app['shap_xgboost_J1.csv'].iloc[position]
contributions = pd.DataFrame({'Variable': X_test_arbres_J1.columns, 'Valeur': X_test_arbres_J1.values[position],
                              'Contribution': shap_ligne[X_test_arbres_J1.columns].values})
fig = graphique_contribution(contributions, shap_ligne['Valeur de base'], 10)
fig.write_image(dossier_figures + 'graphique_contribution.png', width=1200)
fig.show()

In [60]:
# CELL 60: Prédiction : analyse de sensibilité (exemple : Humidity3pm)

# Modèle de démonstration XGBoost J+1 : données non mises à l'échelle, colonnes du jeu J+1
demo = {'modele': modeles_J1['XGBoost'], 'artefacts': artefacts, 'mise_a_echelle': False,
        'colonnes': artefacts['colonnes_J1']}
courbe = calculer_sensibilite(observation, demo, 'Humidity3pm', app['plages_variables.csv'])
fig = graphique_sensibilite(courbe, 'Humidity3pm', seuil_xgboost, observation['Humidity3pm'].values[0])
fig.write_image(dossier_figures + 'graphique_sensibilite.png', width=1200)
fig.show()